<a href="https://colab.research.google.com/github/whitestones011/deep_learning/blob/colab/finetunning_improve_accuracy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from transformers import pipeline
import torch

In [ ]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
prompt = """\
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Please write a birthday card for my good friend Andrew\
<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=200):
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    input_ids = tokenizer.encode(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_new_tokens,
    )
    output = model.generate(input_ids)
    return print(tokenizer.decode(output[0]))

In [ ]:
input_ids = tokenizer.encode(
  prompt,
  return_tensors="pt",
  truncation=True,
  max_length=200,
)

In [ ]:
input_ids

In [ ]:
output = model.generate(input_ids)

In [ ]:
output

In [ ]:
print(tokenizer.decode(output[0]))

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)


In [ ]:
outputs = pipeline(
    prompt,
    max_new_tokens=256,
)

In [ ]:
print(outputs[0]["generated_text"])

# Improved prompt

In [ ]:
prompt2 = (
    "<|begin_of_text|>"  # Start of prompt
    "<|start_header_id|>system<|end_header_id|>\n\n"  #  header - system
    "You are a helpful assistant."  # system prompt
    "<|eot_id|>" # end of turn
    "<|start_header_id|>user<|end_header_id|>\n\n" # header - user
    "Please write a birthday card for my good friend Andrew"
    "<|eot_id|>" # end of turn
    "<|start_header_id|>assistant<|end_header_id|>\n\n" # header - assistant
    )
print(prompt2)

In [ ]:
def make_prompt(user, system=""):
    system_prompt = ""
    if system != "":
        system_prompt = (
            f"<|start_header_id|>system<|end_header_id|>\n\n{system}"
            f"<|eot_id|>"
        )
    prompt = (f"<|begin_of_text|>{system_prompt}"
              f"<|start_header_id|>user<|end_header_id|>\n\n"
              f"{user}"
              f"<|eot_id|>"
              f"<|start_header_id|>assistant<|end_header_id|>\n\n"
         )
    return prompt

In [ ]:
system_prompt = "You are a helpful assistant."
user_prompt = "Please write a birthday card for my good friend Andrew"

prompt3 = make_prompt(user_prompt, system_prompt)

In [ ]:
prompt3 == prompt2

In [ ]:
print(prompt3)

In [ ]:
generate(model, tokenizer, prompt3)

In [ ]:
question = (
    "Tell me a joke about birthday cake"
    )
prompt4 = make_prompt(question)
generate(model, tokenizer, prompt4, max_new_tokens=100)

# SQL agent

In [ ]:
def get_schema():
    return """\
0|Team|TEXT eg. "Toronto Raptors"
1|NAME|TEXT eg. "Otto Porter Jr."
2|Jersey|TEXT eg. "0" and when null has a value "NA"
3|POS|TEXT eg. "PF"
4|AGE|INT eg. "22" in years
5|HT|TEXT eg. `6' 7"` or `6' 10"`
6|WT|TEXT eg. "232 lbs"
7|COLLEGE|TEXT eg. "Michigan" and when null has a value "--"
8|SALARY|TEXT eg. "$9,945,830" and when null has a value "--"
"""

In [ ]:
user = """Who is the highest paid NBA player?"""

In [ ]:
system = f"""You are an NBA analyst with 15 years of experience writing complex SQL queries. Consider the nba_roster table with the following schema:
{get_schema()}

Write a sqlite query to answer the following question. Follow instructions exactly"""

In [ ]:
prompt = make_prompt(user, system)

In [ ]:
print(prompt)

In [ ]:
generate(model, tokenizer, prompt, max_new_tokens=1000)

In [ ]:
import asyncio

async def say_hello_async():
    await asyncio.sleep(2)  # Simulates waiting for 2 seconds
    print("Hello, Async World!")


In [ ]:
import asyncio

async def say_hello_async():
    await asyncio.sleep(2)  # Simulates waiting for 2 seconds
    print("Hello, Async World!")

async def do_something_else():
    print("Starting another task...")
    await asyncio.sleep(1)  # Simulates doing something else for 1 second
    print("Finished another task!")

async def main():
    # Schedule both tasks to run concurrently
    await asyncio.gather(
        say_hello_async(),
        do_something_else(),
    )

In [ ]:
await main()

In [ ]:
async def sleep(x):
  time.sleep(x)

async def say_hello_async():
    await sleep(3)
    print("Hello, Async World!")

async def say_bye_async():
    print("Bye, Async World!")

async def main():
  await asyncio.gather(
      say_hello_async(),
      say_bye_async())

await main()